# 🎯 Goal-Aware Activity Recognition System
## Google Colab Environment Setup

**Project:** Goal-Aware, Activity-Driven Personalized Nutrition Recommendation System

This notebook configures the Colab environment to work with your FYP project stored in OneDrive.

---
### 📋 Table of Contents
1. [OneDrive Connection](#onedrive-connection)
2. [Install Dependencies](#install-dependencies)
3. [Data Loading Utilities](#data-loading)
4. [Verify Setup](#verify-setup)

<a name="onedrive-connection"></a>
## 1. 🔗 OneDrive Connection

Since your project is synced via OneDrive, we'll mount Google Drive and sync your OneDrive folder or upload the dataset directly.

### Option A: Direct Upload (Recommended for Datasets)
Upload your WISDM dataset folder to Colab's runtime storage.

In [ ]:
import os

# 1. Use your actual Windows username 
# (e.g., C:/Users/YourName/Desktop/...)
# Note: Use forward slashes (/) or double backslashes (\\) in Python strings
DRIVE_PROJECT_PATH = os.getcwd()

# 2. Set your paths locally
DATASET_PATH = os.path.join(DRIVE_PROJECT_PATH, 'Datasets/wisdm-dataset')
MODELS_PATH = os.path.join(DRIVE_PROJECT_PATH, 'Models')

# 3. Quick check to make sure the folder exists
if os.path.exists(DRIVE_PROJECT_PATH):
    print(f"Successfully connected to local path: {DRIVE_PROJECT_PATH}")
else:
    print("Error: Path not found. Check your Windows username and folder name.")

Successfully connected to local path: c:\Users\joshw\OneDrive\Desktop\FYP_Work


In [ ]:
# # Mount Google Drive
# from google.colab import drive
# import os

# # The 'force_remount' helps bypass the error you got
# drive.mount('/content/drive', force_remount=True)

# # Set paths
# DRIVE_PROJECT_PATH = '/content/drive/MyDrive/Colab Notebooks/FYP_Work'
# DATASET_PATH = os.path.join(DRIVE_PROJECT_PATH, 'Datasets/wisdm-dataset')
# MODELS_PATH = os.path.join(DRIVE_PROJECT_PATH, 'Models')

# if os.path.exists(DATASET_PATH):
#     print("Found your Drive folder!")
# else:
#     print("Drive mounted, but the subfolder path is wrong.")

In [ ]:
# Option C: Clone from GitHub (if you have a repo setup)
# !git clone https://github.com/YOUR_USERNAME/FYP_Work.git
# DATASET_PATH = '/content/FYP_Work/Datasets/wisdm-dataset'
# MODELS_PATH = '/content/FYP_Work/Models'

### Set Working Paths
Configure the paths based on which option you chose above.

In [ ]:
import os

# ============================================
# 📁 PATH CONFIGURATION - UPDATE AS NEEDED
# ============================================

# # If using Option A (direct upload):
# DATASET_PATH = '/content/Datasets/wisdm-dataset'
# MODELS_PATH = '/content/Models'

# If using Option B (Google Drive mount):
# DATASET_PATH = '/content/drive/MyDrive/Colab Notebooks/FYP_Work/Datasets/wisdm-dataset'
# MODELS_PATH = '/content/drive/MyDrive/Colab Notebooks/FYP_Work/Models'

# Create Models directory if it doesn't exist
os.makedirs(MODELS_PATH, exist_ok=True)

# WISDM Dataset Paths
PHONE_ACCEL_PATH = f'{DATASET_PATH}/raw/phone/accel'
PHONE_GYRO_PATH = f'{DATASET_PATH}/raw/phone/gyro'
WATCH_ACCEL_PATH = f'{DATASET_PATH}/raw/watch/accel'
WATCH_GYRO_PATH = f'{DATASET_PATH}/raw/watch/gyro'

print("📁 Configured Paths:")
print(f"   Dataset Root: {DATASET_PATH}")
print(f"   Models: {MODELS_PATH}")
print(f"   Phone Accel: {PHONE_ACCEL_PATH}")
print(f"   Phone Gyro: {PHONE_GYRO_PATH}")
print(f"   Watch Accel: {WATCH_ACCEL_PATH}")
print(f"   Watch Gyro: {WATCH_GYRO_PATH}")

📁 Configured Paths:
   Dataset Root: c:\Users\joshw\OneDrive\Desktop\FYP_Work\Datasets/wisdm-dataset
   Models: c:\Users\joshw\OneDrive\Desktop\FYP_Work\Models
   Phone Accel: c:\Users\joshw\OneDrive\Desktop\FYP_Work\Datasets/wisdm-dataset/raw/phone/accel
   Phone Gyro: c:\Users\joshw\OneDrive\Desktop\FYP_Work\Datasets/wisdm-dataset/raw/phone/gyro
   Watch Accel: c:\Users\joshw\OneDrive\Desktop\FYP_Work\Datasets/wisdm-dataset/raw/watch/accel
   Watch Gyro: c:\Users\joshw\OneDrive\Desktop\FYP_Work\Datasets/wisdm-dataset/raw/watch/gyro


<a name="install-dependencies"></a>
## 2. 📦 Install Dependencies

Install all required packages for:
- **ML Models**: XGBoost, LightGBM, scikit-learn
- **Explainability**: SHAP
- **Data Processing**: pandas, numpy, scipy

In [ ]:
%%capture
# Install required packages (suppress output for cleaner display)
%pip install xgboost lightgbm shap scikit-learn pandas numpy scipy matplotlib seaborn tqdm joblib

In [ ]:
# Import all required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.fft import fft
import os
import glob
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import xgboost as xgb
import lightgbm as lgb
import shap
import joblib

print("All libraries imported successfully!")

# Display versions
print(f"\n Package Versions:")
print(f"   NumPy: {np.__version__}")
print(f"   Pandas: {pd.__version__}")
print(f"   XGBoost: {xgb.__version__}")
print(f"   LightGBM: {lgb.__version__}")
print(f"   SHAP: {shap.__version__}")

All libraries imported successfully!

 Package Versions:
   NumPy: 2.4.1
   Pandas: 2.3.3
   XGBoost: 3.1.3
   LightGBM: 4.6.0
   SHAP: 0.50.0


<a name="data-loading"></a>
## 3. 📊 Data Loading Utilities

Functions to load and parse the WISDM dataset format:
- Format: `subject_id, activity_code, timestamp, x, y, z;`
- Activities: A (Walking), B (Jogging), C (Stairs), D (Sitting), E (Standing), etc.

In [ ]:
# WISDM Activity Labels Mapping
ACTIVITY_LABELS = {
    'A': 'Walking',
    'B': 'Jogging', 
    'C': 'Stairs',
    'D': 'Sitting',
    'E': 'Standing',
    'F': 'Typing',
    'G': 'Brushing Teeth',
    'H': 'Eating Soup',
    'I': 'Eating Chips',
    'J': 'Eating Pasta',
    'K': 'Drinking',
    'L': 'Eating Sandwich',
    'M': 'Kicking',
    'O': 'Catching',
    'P': 'Dribbling',
    'Q': 'Writing',
    'R': 'Clapping',
    'S': 'Folding'
}

# Primary activities for your project (matching MET mapping)
PRIMARY_ACTIVITIES = ['A', 'B', 'C', 'D', 'E']  # Walking, Jogging, Sitting, Standing

print("WISDM Activity Labels:")
for code, name in ACTIVITY_LABELS.items():
    marker = "⭐" if code in PRIMARY_ACTIVITIES else "  "
    print(f"   {marker} {code}: {name}")

WISDM Activity Labels:
   ⭐ A: Walking
   ⭐ B: Jogging
   ⭐ C: Stairs
   ⭐ D: Sitting
   ⭐ E: Standing
      F: Typing
      G: Brushing Teeth
      H: Eating Soup
      I: Eating Chips
      J: Eating Pasta
      K: Drinking
      L: Eating Sandwich
      M: Kicking
      O: Catching
      P: Dribbling
      Q: Writing
      R: Clapping
      S: Folding


In [ ]:
def load_wisdm_file(filepath):
    """
    Load a single WISDM data file.
    
    Args:
        filepath: Path to the data file
        
    Returns:
        DataFrame with columns: subject_id, activity, timestamp, x, y, z
    """
    data = []
    
    with open(filepath, 'r') as f:
        for line in f:
            # Remove trailing semicolon and newline
            line = line.strip().rstrip(';')
            if not line:
                continue
            
            try:
                parts = line.split(',')
                if len(parts) >= 6:
                    subject_id = int(parts[0])
                    activity = parts[1]
                    timestamp = int(parts[2])
                    x = float(parts[3])
                    y = float(parts[4])
                    z = float(parts[5])
                    data.append([subject_id, activity, timestamp, x, y, z])
            except (ValueError, IndexError):
                continue
    
    return pd.DataFrame(data, columns=['subject_id', 'activity', 'timestamp', 'x', 'y', 'z'])


def load_wisdm_sensor_data(sensor_path, sensor_type='accel', device='phone'):
    """
    Load all WISDM data files for a specific sensor and device.
    
    Args:
        sensor_path: Path to the sensor data folder
        sensor_type: 'accel' or 'gyro'
        device: 'phone' or 'watch'
        
    Returns:
        DataFrame with all sensor data
    """
    all_files = glob.glob(os.path.join(sensor_path, '*.txt'))
    
    if not all_files:
        print(f"No files found in {sensor_path}")
        return pd.DataFrame()
    
    dfs = []
    for filepath in tqdm(all_files, desc=f"Loading {device} {sensor_type}"):
        df = load_wisdm_file(filepath)
        if not df.empty:
            df['sensor'] = sensor_type
            df['device'] = device
            dfs.append(df)
    
    if dfs:
        return pd.concat(dfs, ignore_index=True)
    return pd.DataFrame()


def load_all_wisdm_data():
    """
    Load all WISDM sensor data (phone + watch, accel + gyro).
    
    Returns:
        Dictionary with keys: 'phone_accel', 'phone_gyro', 'watch_accel', 'watch_gyro'
    """
    data = {}
    
    print("Loading WISDM Dataset...\n")
    
    # Phone data
    data['phone_accel'] = load_wisdm_sensor_data(PHONE_ACCEL_PATH, 'accel', 'phone')
    data['phone_gyro'] = load_wisdm_sensor_data(PHONE_GYRO_PATH, 'gyro', 'phone')
    
    # Watch data
    data['watch_accel'] = load_wisdm_sensor_data(WATCH_ACCEL_PATH, 'accel', 'watch')
    data['watch_gyro'] = load_wisdm_sensor_data(WATCH_GYRO_PATH, 'gyro', 'watch')
    
    print("\nData Loading Complete!")
    print("\nDataset Summary:")
    for key, df in data.items():
        if not df.empty:
            print(f"   {key}: {len(df):,} samples, {df['subject_id'].nunique()} subjects")
    
    return data

print("Data loading functions defined!")

Data loading functions defined!


In [ ]:
def create_sliding_windows(df, window_size=200, step_size=100):
    """
    Create sliding windows from sensor data.
    
    Args:
        df: DataFrame with x, y, z columns
        window_size: Number of samples per window (200 = ~4 seconds at 50Hz)
        step_size: Step between windows (100 = 50% overlap)
        
    Returns:
        windows: numpy array of shape (n_windows, window_size, 3)
        labels: numpy array of activity labels
        subjects: numpy array of subject IDs
    """
    windows = []
    labels = []
    subjects = []
    
    # Process each subject separately
    for subject_id in df['subject_id'].unique():
        subject_data = df[df['subject_id'] == subject_id]
        
        # Process each activity separately
        for activity in subject_data['activity'].unique():
            activity_data = subject_data[subject_data['activity'] == activity]
            activity_data = activity_data.sort_values('timestamp')
            
            xyz = activity_data[['x', 'y', 'z']].values
            
            # Create windows
            for start in range(0, len(xyz) - window_size, step_size):
                window = xyz[start:start + window_size]
                windows.append(window)
                labels.append(activity)
                subjects.append(subject_id)
    
    return np.array(windows), np.array(labels), np.array(subjects)

print("Sliding window function defined!")

Sliding window function defined!


In [ ]:
def extract_features(windows):

    features = []
    
    for window in tqdm(windows, desc="Extracting features"):
        feat = {}
        
        for i, axis in enumerate(['x', 'y', 'z']):
            data = window[:, i]
            
            # Time domain features
            feat[f'{axis}_mean'] = np.mean(data)
            feat[f'{axis}_std'] = np.std(data)
            feat[f'{axis}_min'] = np.min(data)
            feat[f'{axis}_max'] = np.max(data)
            feat[f'{axis}_range'] = np.max(data) - np.min(data)
            feat[f'{axis}_median'] = np.median(data)
            feat[f'{axis}_mad'] = np.median(np.abs(data - np.median(data)))
            feat[f'{axis}_iqr'] = np.percentile(data, 75) - np.percentile(data, 25)
            feat[f'{axis}_skew'] = stats.skew(data)
            feat[f'{axis}_kurtosis'] = stats.kurtosis(data)
            feat[f'{axis}_rms'] = np.sqrt(np.mean(data**2))
            feat[f'{axis}_zero_crossings'] = np.sum(np.diff(np.sign(data)) != 0)
            
            # Frequency domain features (FFT)
            fft_vals = np.abs(fft(data))
            fft_vals = fft_vals[:len(fft_vals)//2]
            
            feat[f'{axis}_fft_mean'] = np.mean(fft_vals)
            feat[f'{axis}_fft_std'] = np.std(fft_vals)
            feat[f'{axis}_fft_max'] = np.max(fft_vals)
            feat[f'{axis}_fft_energy'] = np.sum(fft_vals**2) / len(fft_vals)
            feat[f'{axis}_fft_entropy'] = stats.entropy(fft_vals + 1e-10)
        
        # Magnitude features
        magnitude = np.sqrt(np.sum(window**2, axis=1))
        feat['mag_mean'] = np.mean(magnitude)
        feat['mag_std'] = np.std(magnitude)
        feat['mag_max'] = np.max(magnitude)
        feat['mag_min'] = np.min(magnitude)
        
        # Correlation features
        feat['xy_corr'] = np.corrcoef(window[:, 0], window[:, 1])[0, 1]
        feat['xz_corr'] = np.corrcoef(window[:, 0], window[:, 2])[0, 1]
        feat['yz_corr'] = np.corrcoef(window[:, 1], window[:, 2])[0, 1]
        
        features.append(feat)
    
    return pd.DataFrame(features)

print("Feature extraction function defined!")

Feature extraction function defined!


In [ ]:
def save_model(model, model_name, scaler=None, label_encoder=None):
    """
    Save trained model and preprocessing objects.
    
    Args:
        model: Trained model
        model_name: Name for the saved model
        scaler: StandardScaler object (optional)
        label_encoder: LabelEncoder object (optional)
    """
    model_path = os.path.join(MODELS_PATH, model_name)
    os.makedirs(model_path, exist_ok=True)
    
    # Save model
    joblib.dump(model, os.path.join(model_path, 'model.joblib'))
    
    # Save scaler if provided
    if scaler:
        joblib.dump(scaler, os.path.join(model_path, 'scaler.joblib'))
    
    # Save label encoder if provided
    if label_encoder:
        joblib.dump(label_encoder, os.path.join(model_path, 'label_encoder.joblib'))
    
    print(f"✅ Model saved to: {model_path}")


def load_model(model_name):
    """
    Load a saved model and preprocessing objects.
    
    Args:
        model_name: Name of the saved model
        
    Returns:
        model, scaler, label_encoder
    """
    model_path = os.path.join(MODELS_PATH, model_name)
    
    model = joblib.load(os.path.join(model_path, 'model.joblib'))
    
    scaler_path = os.path.join(model_path, 'scaler.joblib')
    scaler = joblib.load(scaler_path) if os.path.exists(scaler_path) else None
    
    le_path = os.path.join(model_path, 'label_encoder.joblib')
    label_encoder = joblib.load(le_path) if os.path.exists(le_path) else None
    
    print(f"Model loaded from: {model_path}")
    return model, scaler, label_encoder

print("Model save/load functions defined!")

Model save/load functions defined!


<a name="verify-setup"></a>
## 4. ✅ Verify Setup

Run this cell to verify that everything is configured correctly.

In [ ]:
def verify_setup():
    """
    Verify that the Colab environment is properly configured.
    """
    print("Verifying Setup...\n")
    
    checks = {
        'Dataset Path Exists': os.path.exists(DATASET_PATH),
        'Phone Accel Folder': os.path.exists(PHONE_ACCEL_PATH),
        'Phone Gyro Folder': os.path.exists(PHONE_GYRO_PATH),
        'Watch Accel Folder': os.path.exists(WATCH_ACCEL_PATH),
        'Watch Gyro Folder': os.path.exists(WATCH_GYRO_PATH),
        'Models Folder': os.path.exists(MODELS_PATH),
    }
    
    all_passed = True
    for check, passed in checks.items():
        status = "✅" if passed else "❌"
        print(f"   {status} {check}")
        if not passed:
            all_passed = False
    
    if all_passed:
        print("\n🎉 All checks passed! Environment is ready.")
        
        # Count files
        phone_accel_files = len(glob.glob(os.path.join(PHONE_ACCEL_PATH, '*.txt')))
        phone_gyro_files = len(glob.glob(os.path.join(PHONE_GYRO_PATH, '*.txt')))
        watch_accel_files = len(glob.glob(os.path.join(WATCH_ACCEL_PATH, '*.txt')))
        watch_gyro_files = len(glob.glob(os.path.join(WATCH_GYRO_PATH, '*.txt')))
        
        print(f"\nData Files Found:")
        print(f"   Phone Accelerometer: {phone_accel_files} files")
        print(f"   Phone Gyroscope: {phone_gyro_files} files")
        print(f"   Watch Accelerometer: {watch_accel_files} files")
        print(f"   Watch Gyroscope: {watch_gyro_files} files")
    else:
        print("\nSome checks failed. Please verify your data paths.")

# Run verification
verify_setup()

Verifying Setup...

   ✅ Dataset Path Exists
   ✅ Phone Accel Folder
   ✅ Phone Gyro Folder
   ✅ Watch Accel Folder
   ✅ Watch Gyro Folder
   ✅ Models Folder

🎉 All checks passed! Environment is ready.

Data Files Found:
   Phone Accelerometer: 51 files
   Phone Gyroscope: 51 files
   Watch Accelerometer: 51 files
   Watch Gyroscope: 51 files


---
## 🚀 Quick Start Example

After verification passes, you can start working with the data:

In [ ]:
# Example: Load a sample file and preview the data
sample_file = os.path.join(PHONE_ACCEL_PATH, 'data_1600_accel_phone.txt')

if os.path.exists(sample_file):
    sample_df = load_wisdm_file(sample_file)
    print("📊 Sample Data Preview:")
    print(f"   Shape: {sample_df.shape}")
    print(f"   Columns: {list(sample_df.columns)}")
    print(f"   Activities: {sample_df['activity'].unique()}")
    print(f"\n{sample_df.head(10)}")
else:
    print("⚠️ Sample file not found. Please upload the dataset first.")

📊 Sample Data Preview:
   Shape: (64311, 6)
   Columns: ['subject_id', 'activity', 'timestamp', 'x', 'y', 'z']
   Activities: ['A' 'B' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'J' 'K' 'L' 'M' 'O' 'P' 'Q' 'R' 'S']

   subject_id activity        timestamp         x          y         z
0        1600        A  252207666810782 -0.364761   8.793503  1.055084
1        1600        A  252207717164786 -0.879730   9.768784  1.016998
2        1600        A  252207767518790  2.001495  11.109070  2.619156
3        1600        A  252207817872794  0.450623  12.651642  0.184555
4        1600        A  252207868226798 -2.164352  13.928436 -4.422485
5        1600        A  252207918580802 -4.332779  13.361191 -0.718872
6        1600        A  252207968934806 -0.319443  13.318359 -0.232025
7        1600        A  252208019288809  1.566452   9.515274 -0.017776
8        1600        A  252208069642813 -0.323746   5.262665  0.322342
9        1600        A  252208119996817 -1.811676   3.710510  1.373932


---
## 📚 Next Steps

1. **Data Exploration**: Analyze activity distributions and sensor patterns
2. **Feature Engineering**: Create sliding windows and extract features
3. **Model Training**: Train Random Forest, XGBoost, and LightGBM models
4. **Sensor Fusion**: Combine phone and watch data for improved accuracy
5. **Explainability**: Use SHAP to interpret model predictions

---
*FYP: Goal-Aware, Activity-Driven Personalized Nutrition Recommendation System*